In [ ]:
# Cell 1
!pip install qrcode[pil] Pillow numpy

# Cell 2 — update your URL
URL = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"

In [ ]:
import torch
import qrcode
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
import os

# -----------------------------
# Settings
# -----------------------------

MODEL_CACHE = "./models"
OUTPUT_DIR = "./output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT = """
premium cooking oil advertisement,
olive oil bottle with olives and avocado,
green natural background,
qr code hidden in artwork
"""

NEGATIVE_PROMPT = "blurry, distorted qr, broken pattern"

QR_CONTENT = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"

NUM_IMAGES = 4

GUIDANCE = 7
CONTROL_SCALE = 1.8
STEPS = 40

BOX_SIZE = 16
BORDER = 6

# -----------------------------
# Load Model (only once)
# -----------------------------

print("Loading ControlNet...")

controlnet = ControlNetModel.from_pretrained(
    "monster-labs/control_v1p_sd15_qrcode_monster",
    cache_dir=MODEL_CACHE,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    cache_dir=MODEL_CACHE,
    safety_checker=None,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

if DEVICE == "cuda":
    pipe.enable_xformers_memory_efficient_attention()

# -----------------------------
# Generate QR Base
# -----------------------------

qr = qrcode.QRCode(
    version=1,
    error_correction=qrcode.constants.ERROR_CORRECT_H,
    box_size=BOX_SIZE,
    border=BORDER
)

qr.add_data(QR_CONTENT)
qr.make(fit=True)

qr_img = qr.make_image(fill_color="black", back_color="white")

# Resize for SD
w, h = qr_img.size
w = (w + 255) // 256 * 256
h = (h + 255) // 256 * 256

bg = Image.new("L", (w, h), 128)

coords = ((w - qr_img.size[0]) // 2, (h - qr_img.size[1]) // 2)
bg.paste(qr_img.convert("L"), coords)

# -----------------------------
# Generate Images
# -----------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

for i in range(NUM_IMAGES):

    print(f"Generating image {i+1}/{NUM_IMAGES}")

    generator = torch.Generator(DEVICE).manual_seed(torch.randint(0,999999,(1,)).item())

    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=bg,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=CONTROL_SCALE,
        num_inference_steps=STEPS,
        generator=generator
    ).images[0]

    path = f"{OUTPUT_DIR}/qr_art_{i}.png"
    image.save(path)

    print("Saved:", path)

print("Done.")

In [1]:
pip install pyzbar

   ---------------------------------------- 0.0/817.4 kB ? eta -:--:--
   --------------------------------------- 817.4/817.4 kB 17.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
Patri AI Cooking Oil — 7 Blend-Themed Scannable QR Codes
=========================================================
Blend themes match the exact ratio table:

  Heart Friendly    — High-Oleic Sunflower(337) + EVOO(187) + Rice Bran(112) + Flaxseed(37) + Walnut(75)
  Adult Family      — High-Oleic Sunflower(225) + Rice Bran(225) + Sesame(75) + Coconut(37) + Groundnut(188)
  Family Balanced   — High-Oleic Sunflower(300) + Rice Bran(225) + Sesame(75) + Coconut(37) + Groundnut(112)
  Healthy Aging     — High-Oleic Sunflower(112) + Flaxseed(112) + Walnut(75) + Avocado(260) + Olive(187)
  Fat Switch Oil    — High-Oleic Sunflower(187) + Rice Bran(75) + Flaxseed(150) + Sesame(37) + Olive(187) + Fish(75) + Mustard(37)
  Heat Stable       — Rice Bran(227) + Sesame(110) + Coconut(37) + Groundnut(300) + Palmolein(75)
  Guacamole Oil     — Sesame(75) + Avocado(375) + Corn(300)

Settings proven to scan from phone:
  CONTROL_SCALE = 1.8, gray canvas, logo dark-bg stripped

Run:
  python patri_qr_blends_v4.py
"""

import os, random, torch, qrcode
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import cv2

# ══════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════
MODEL_CACHE = "./models"
OUTPUT_DIR  = "./patri_blends_output"
LOGO_PATH   = "C:/Users/pkitt/Downloads/Patri Image.png"

QR_CONTENT  = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"
MAX_RETRIES = 10

CONTROL_SCALE = 1.8    # proven working value — do not lower
GUIDANCE      = 7.0
STEPS         = 40
BOX_SIZE      = 16
BORDER        = 6

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# ══════════════════════════════════════════════════════════════
# 7 BLEND-THEMED PROMPTS
# Each prompt is designed around the dominant oils in that blend.
# Dominant = highest ml value(s) in the ratio table.
# ══════════════════════════════════════════════════════════════
BLEND_STYLES = [
    {
        # High-Oleic Sunflower 337 + EVOO 187 + Rice Bran 112 + Flaxseed 37 + Walnut 75
        "name": "heart_friendly",
        "label": "HEART FRIENDLY",
        "prompt": (
            "overhead flat lay macro of sunflower petals and seeds, fresh green olive branches, "
            "golden olive oil drops, flaxseeds and walnut halves arranged on dark slate surface, "
            "premium Patri heart-friendly cooking oil blend, warm gold and green palette, "
            "studio lighting, photorealistic 8k, elegant minimal composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, butter",
    },
    {
        # High-Oleic Sunflower 225 + Rice Bran 225 + Sesame 75 + Coconut 37 + Groundnut 188
        "name": "adult_family",
        "label": "ADULT FAMILY",
        "prompt": (
            "overhead flat lay macro of golden sunflower seeds, rice grains and bran, "
            "white and black sesame seeds, shelled peanuts and coconut shavings on dark stone surface, "
            "premium Patri adult family cooking oil blend, warm ivory and amber tones, "
            "studio lighting, photorealistic 8k, balanced elegant composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, cooked food",
    },
    {
        # High-Oleic Sunflower 300 + Rice Bran 225 + Sesame 75 + Coconut 37 + Groundnut 112
        "name": "family_balanced",
        "label": "FAMILY BALANCED",
        "prompt": (
            "overhead flat lay macro of bright sunflower petals and seeds, raw rice bran, "
            "sesame seeds, fresh coconut pieces and raw peanuts on dark marble surface, "
            "premium Patri family balanced cooking oil blend, vibrant yellow and natural tones, "
            "studio lighting, photorealistic 8k, harmonious flat lay composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, processed food",
    },
    {
        # High-Oleic Sunflower 112 + Flaxseed 112 + Walnut 75 + Avocado 260 + Olive 187
        "name": "healthy_aging",
        "label": "HEALTHY AGING",
        "prompt": (
            "overhead flat lay macro of fresh halved avocados, green olive branches, "
            "golden flaxseeds, walnut halves and sunflower seeds on dark slate surface, "
            "premium Patri healthy aging cooking oil blend, rich green and gold tones, "
            "studio lighting, photorealistic 8k, luxurious elegant composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, guacamole",
    },
    {
        # High-Oleic Sunflower 187 + Rice Bran 75 + Flaxseed 150 + Sesame 37 + Olive 187 + Fish 75 + Mustard 37
        "name": "fat_switch",
        "label": "FAT SWITCH OIL",
        "prompt": (
            "overhead flat lay macro of golden flaxseeds, sunflower seeds, olive branches, "
            "yellow mustard seeds and sesame seeds scattered on dark textured surface, "
            "premium Patri fat switch cooking oil blend, vibrant gold and yellow palette, "
            "studio lighting, photorealistic 8k, dynamic flat lay composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, fish, sauce",
    },
    {
        # Rice Bran 227 + Sesame 110 + Coconut 37 + Groundnut 300 + Palmolein 75
        "name": "heat_stable",
        "label": "HEAT STABLE",
        "prompt": (
            "overhead flat lay macro of raw peanuts and groundnuts, rice grains and bran, "
            "sesame seeds, fresh coconut pieces on dark stone surface, "
            "premium Patri heat stable cooking oil blend, warm earthy amber and ivory palette, "
            "studio lighting, photorealistic 8k, bold minimal composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, roasted nuts",
    },
    {
        # Sesame 75 + Avocado 375 + Corn 300
        "name": "guacamole_oil",
        "label": "GUACAMOLE OIL",
        "prompt": (
            "overhead flat lay macro of fresh halved avocados with vivid green flesh, "
            "golden corn kernels and corn cob slices, white sesame seeds on dark obsidian surface, "
            "premium Patri guacamole cooking oil blend, vibrant green and golden yellow palette, "
            "studio lighting, photorealistic 8k, striking bold composition"
        ),
        "negative": "blurry, text, watermark, face, distorted, low quality, cartoon, guacamole dip",
    },
]


# ══════════════════════════════════════════════════════════════
# QR CONTROL IMAGE  (gray canvas — proven approach)
# ══════════════════════════════════════════════════════════════

def make_qr_control(content: str) -> Image.Image:
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=BOX_SIZE,
        border=BORDER,
    )
    qr.add_data(content)
    qr.make(fit=True)
    qr_img = qr.make_image(fill_color="black", back_color="white")

    w, h = qr_img.size
    w = (w + 255) // 256 * 256
    h = (h + 255) // 256 * 256

    # Mid-gray canvas (128) — gives ControlNet tonal room to blend
    canvas = Image.new("L", (w, h), 128)
    ox = (w - qr_img.size[0]) // 2
    oy = (h - qr_img.size[1]) // 2
    canvas.paste(qr_img.convert("L"), (ox, oy))
    return canvas.convert("RGB")


# ══════════════════════════════════════════════════════════════
# SCANNER  (multi-pass + multi-scale)
# ══════════════════════════════════════════════════════════════

_detector = cv2.QRCodeDetector()


def _cv_decode(pil_img: Image.Image, expected: str) -> bool:
    arr  = np.array(pil_img.convert("RGB"))
    bgr  = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    for img_in in [bgr, gray]:
        d, _, _ = _detector.detectAndDecode(img_in)
        if d and d.strip() == expected.strip():
            return True

    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    d, _, _ = _detector.detectAndDecode(thresh)
    if d and d.strip() == expected.strip():
        return True

    adapt = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    d, _, _ = _detector.detectAndDecode(adapt)
    return bool(d and d.strip() == expected.strip())


def is_scannable(img: Image.Image, expected: str) -> bool:
    if _cv_decode(img, expected):
        return True
    enhanced = ImageEnhance.Contrast(img).enhance(2.0)
    if _cv_decode(enhanced, expected):
        return True
    for scale in (0.75, 0.5, 1.25):
        r = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)
        if _cv_decode(r, expected):
            return True
        if _cv_decode(ImageEnhance.Contrast(r).enhance(2.0), expected):
            return True
    return False


# ══════════════════════════════════════════════════════════════
# LOGO & LABEL HELPERS
# ══════════════════════════════════════════════════════════════

def strip_dark_background(logo: Image.Image, threshold: int = 60) -> Image.Image:
    """Make dark/black logo background pixels transparent."""
    rgba = logo.convert("RGBA")
    arr  = np.array(rgba, dtype=np.uint8)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    dark_mask = (r < threshold) & (g < threshold) & (b < threshold)
    arr[dark_mask, 3] = 0
    return Image.fromarray(arr, "RGBA")


def add_logo_centre(img: Image.Image, logo_path: str,
                    size_pct: float = 0.18) -> Image.Image:
    if not os.path.exists(logo_path):
        print(f"   ℹ  Logo not found: {logo_path} — skipping")
        return img

    logo = Image.open(logo_path).convert("RGBA")
    logo = strip_dark_background(logo, threshold=60)

    max_dim = int(img.width * size_pct)
    logo.thumbnail((max_dim, max_dim), Image.LANCZOS)
    lw, lh = logo.size

    cx  = (img.width  - lw) // 2
    cy  = (img.height - lh) // 2
    pad = 10

    bs      = (lw + pad * 2, lh + pad * 2)
    backing = Image.new("RGBA", bs, (255, 255, 255, 255))
    mask    = Image.new("L",    bs, 0)
    ImageDraw.Draw(mask).rounded_rectangle(
        [0, 0, bs[0] - 1, bs[1] - 1], radius=12, fill=255
    )
    backing.putalpha(mask)

    out = img.convert("RGBA")
    out.paste(backing, (cx - pad, cy - pad), backing)
    out.paste(logo,    (cx, cy),             logo)
    return out.convert("RGB")


def add_blend_label(img: Image.Image, label: str) -> Image.Image:
    out     = img.convert("RGBA")
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    draw    = ImageDraw.Draw(overlay)
    bar_h   = 40
    # Dark green bar
    draw.rectangle([0, img.height - bar_h, img.width, img.height],
                   fill=(15, 50, 15, 210))
    # Label text in gold
    draw.text((img.width // 2, img.height - bar_h // 2),
              label, fill=(255, 215, 60, 255), anchor="mm")
    return Image.alpha_composite(out, overlay).convert("RGB")


# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

def main():
    print(f"🖥  Device : {DEVICE}")
    print(f"⚙  control_scale={CONTROL_SCALE}  guidance={GUIDANCE}  steps={STEPS}")
    print(f"📡 Scanner : OpenCV multi-pass + multi-scale\n")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Build QR control ──────────────────────────────────────
    print("🔲 Building QR control image (gray canvas) …")
    qr_control = make_qr_control(QR_CONTENT)
    qr_control.save(os.path.join(OUTPUT_DIR, "_qr_control_base.png"))
    assert is_scannable(qr_control, QR_CONTENT), "❌ Base QR failed scan check!"
    print("✅ Base QR verified scannable\n")

    # ── Load pipeline ─────────────────────────────────────────
    try:
        from diffusers import (ControlNetModel,
                               StableDiffusionControlNetPipeline,
                               DDIMScheduler)
    except ImportError:
        print("❌ diffusers not installed.")
        print("   pip install torch diffusers transformers accelerate")
        return

    print("📦 Loading ControlNet pipeline …")
    controlnet = ControlNetModel.from_pretrained(
        "monster-labs/control_v1p_sd15_qrcode_monster",
        cache_dir=MODEL_CACHE,
        torch_dtype=DTYPE,
    )
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        cache_dir=MODEL_CACHE,
        safety_checker=None,
        torch_dtype=DTYPE,
    ).to(DEVICE)
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    if DEVICE == "cuda":
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass

    total  = len(BLEND_STYLES)
    saved  = 0
    failed = []

    print(f"\n🌿 Generating {total} blend-themed QR codes …\n")
    print("=" * 60)

    # Print blend summary
    print("Blend ratios loaded:")
    ratios = {
        "heart_friendly":  "HO-Sunflower 337 + EVOO 187 + Rice Bran 112 + Flaxseed 37 + Walnut 75",
        "adult_family":    "HO-Sunflower 225 + Rice Bran 225 + Sesame 75 + Coconut 37 + Groundnut 188",
        "family_balanced": "HO-Sunflower 300 + Rice Bran 225 + Sesame 75 + Coconut 37 + Groundnut 112",
        "healthy_aging":   "HO-Sunflower 112 + Flaxseed 112 + Walnut 75 + Avocado 260 + Olive 187",
        "fat_switch":      "HO-Sunflower 187 + Rice Bran 75 + Flaxseed 150 + Sesame 37 + Olive 187 + Fish 75 + Mustard 37",
        "heat_stable":     "Rice Bran 227 + Sesame 110 + Coconut 37 + Groundnut 300 + Palmolein 75",
        "guacamole_oil":   "Sesame 75 + Avocado 375 + Corn 300",
    }
    for k, v in ratios.items():
        print(f"  {k:20s} → {v}")
    print()

    for idx, style in enumerate(BLEND_STYLES):
        name    = style["name"]
        label   = style["label"]
        prompt  = style["prompt"]
        neg     = style["negative"]
        success = False

        print(f"\n[{idx+1:02d}/{total}] 🫙 {label}")
        print(f"  Prompt: {prompt[:70]}…")

        for attempt in range(MAX_RETRIES):
            seed = random.randint(0, 999_999)
            print(f"  attempt {attempt+1:2d}  seed={seed}  ",
                  end="", flush=True)

            generator = torch.Generator(DEVICE).manual_seed(seed)
            w, h = qr_control.size

            try:
                raw = pipe(
                    prompt=prompt,
                    negative_prompt=neg,
                    image=qr_control,
                    guidance_scale=GUIDANCE,
                    controlnet_conditioning_scale=CONTROL_SCALE,
                    num_inference_steps=STEPS,
                    generator=generator,
                    width=w,
                    height=h,
                ).images[0]
            except Exception as e:
                print(f"ERROR: {e}")
                continue

            # Scan raw output first
            ok_raw = is_scannable(raw, QR_CONTENT)

            # Add logo with dark-bg stripping
            with_logo = add_logo_centre(raw, LOGO_PATH, size_pct=0.18)
            ok_logo   = is_scannable(with_logo, QR_CONTENT)

            if ok_logo:
                print("✅ SCANNABLE (with logo)")
                final    = add_blend_label(with_logo, label)
                out_path = os.path.join(
                    OUTPUT_DIR, f"patri_{idx+1:02d}_{name}_seed{seed}.png"
                )
                final.save(out_path)
                print(f"  💾 Saved → {out_path}")
                saved  += 1
                success = True
                break

            elif ok_raw:
                print("⚠  scannable (no logo — logo too opaque)")
                final    = add_blend_label(raw, label)
                out_path = os.path.join(
                    OUTPUT_DIR, f"patri_{idx+1:02d}_{name}_seed{seed}_nologo.png"
                )
                final.save(out_path)
                print(f"  💾 Saved (no logo) → {out_path}")
                saved  += 1
                success = True
                break

            else:
                print("❌ not scannable")

        if not success:
            failed.append(name)
            print(f"  ⚠  FAILED after {MAX_RETRIES} attempts — skipping {name}")

    print("\n" + "=" * 60)
    print(f"✅ Complete: {saved}/{total} QR codes saved to ./{OUTPUT_DIR}/")
    if failed:
        print(f"⚠  Failed: {', '.join(failed)}")
        print("   Tip: re-run — different random seeds usually succeed")
    print("\nTo fix logo transparency permanently:")
    print("  Open 'Patri Image.png' in Photoshop/GIMP")
    print("  Remove the dark green background layer")
    print("  Export as PNG with transparent background")
    print("  This gives the cleanest logo overlay result")


if __name__ == "__main__":
    main()

🖥  Device : cpu
⚙  control_scale=1.8  guidance=7.0  steps=40
📡 Scanner : OpenCV multi-pass + multi-scale

🔲 Building QR control image (gray canvas) …
✅ Base QR verified scannable

📦 Loading ControlNet pipeline …


C:\Users\pkitt\anaconda3\Lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: models\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information

...

  0%|          | 0/40 [00:00<?, ?it/s]

❌ not scannable
  attempt 10  seed=95935  

  0%|          | 0/40 [00:00<?, ?it/s]

❌ not scannable
  attempt  2  seed=900441  

  0%|          | 0/40 [00:00<?, ?it/s]